# 04 - Indicadores Propuestos: Dimension Actividad Economica**Proyecto:** Indice de Ingresos Operacionales - Cali  **Equipo:** ITT Cali Inteligente - Gobierno de Datos  **Repositorio:** https://github.com/j0rg3c45/Indice_ingresos_operacionales.git## ObjetivoImplementar los indicadores propuestos para la dimension de Actividad Economicadel ITT, basados en la propuesta del profesor Carlos Federico Vallejo.## Subdimensiones implementadas1. Entrada empresarial (parcial)2. Persistencia empresarial (parcial)3. Transformacion sectorial (completo)4. Intensidad economica (completo)5. Empleo formal (parcial)

## 1. Configuracion y carga de datos

In [ ]:
import osimport refrom pathlib import Pathfrom datetime import datetimeimport pandas as pdimport numpy as npimport matplotlibimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport geopandas as gpdimport warningswarnings.filterwarnings('ignore')# Formato abreviado para ejesdef formato_abreviado(x, pos):    '''Convierte: 1000->1K, 1000000->1M, 1000000000->1B'''    if abs(x) >= 1e9: return f"{x/1e9:.1f}B"    elif abs(x) >= 1e6: return f"{x/1e6:.1f}M"    elif abs(x) >= 1e3: return f"{x/1e3:.0f}K"    else: return f"{x:.0f}"def aplicar_formato(ax, eje='y'):    fmt = mticker.FuncFormatter(formato_abreviado)    if eje == 'y': ax.yaxis.set_major_formatter(fmt)    elif eje == 'x': ax.xaxis.set_major_formatter(fmt)# RutasREPO_URL = "https://github.com/j0rg3c45/Indice_ingresos_operacionales.git"REPO_NAME = "Indice_ingresos_operacionales"EN_COLAB = os.path.exists("/content")if EN_COLAB:    WORK_DIR = Path("/content") / REPO_NAME    if not WORK_DIR.exists():        os.system(f"git clone {REPO_URL}")    else:        os.system(f"cd {WORK_DIR} && git pull")else:    WORK_DIR = Path(os.getcwd()).parent    if not (WORK_DIR / "README.md").exists():        WORK_DIR = Path(os.getcwd())DATA_DIR = WORK_DIR / "data"GEO_DIR = DATA_DIR / "info_geo"OUTPUT_DIR = WORK_DIR / "outputs"OUTPUT_DIR.mkdir(parents=True, exist_ok=True)# Cargar datosprint("Cargando Registro Mercantil...")ARCHIVO_EXCEL = list(DATA_DIR.glob("Registro*.xlsx"))[0]df = pd.read_excel(ARCHIVO_EXCEL, engine="openpyxl")df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)print(f"  Registros: {len(df):,}")# Cargar GeoJSON comunasgeojson_files = list(GEO_DIR.glob("**/Comunas.geojson"))gdf_comunas = gpd.read_file(geojson_files[0]) if geojson_files else Noneif gdf_comunas is not None:    gdf_comunas['comuna_key'] = 'Comuna ' + gdf_comunas['comuna'].astype(int).astype(str).str.zfill(2)    # Calcular area en hectareas    gdf_comunas_proj = gdf_comunas.to_crs('EPSG:3116')    gdf_comunas['area_ha'] = gdf_comunas_proj.geometry.area / 10000    print(f"  Comunas GeoJSON: {len(gdf_comunas)}")

## 2. Parseo de fechas

In [ ]:
# Las fechas estan en formato texto: "11 de febrero de 2025"# Necesitamos extraer el ano para calcular antiguedad y tasa de nuevas matriculascol_fecha_mat = [c for c in df.columns if 'fecha' in c and 'matricula' in c][0]col_fecha_ren = [c for c in df.columns if 'fecha' in c and 'renovacion' in c][0]col_comuna = [c for c in df.columns if 'comuna' in c][0]col_ingresos = [c for c in df.columns if 'ingreso' in c][0]col_empleo = [c for c in df.columns if 'personal' in c][0]col_tamano = [c for c in df.columns if 'tama' in c][0]col_ciiu = [c for c in df.columns if 'codigo' in c and 'ciiu' in c][0]col_sector = [c for c in df.columns if 'sector' in c][0]col_hombres = [c for c in df.columns if 'hombre' in c][0]col_mujeres = [c for c in df.columns if 'mujer' in c][0]# Funcion para extraer ano del texto "11 de febrero de 2025"def extraer_ano(texto):    '''Extrae el ano de un texto como "11 de febrero de 2025" -> 2025'''    if pd.isna(texto):        return np.nan    match = re.search(r'(\d{4})', str(texto))    return int(match.group(1)) if match else np.nan# Extraer anosdf['ano_matricula'] = df[col_fecha_mat].apply(extraer_ano)df['ano_renovacion'] = df[col_fecha_ren].apply(extraer_ano)# Convertir numericasdf[col_ingresos] = pd.to_numeric(df[col_ingresos], errors='coerce')df[col_empleo] = pd.to_numeric(df[col_empleo], errors='coerce')# Filtrar con comunadf_cc = df[df[col_comuna].notna()].copy()print(f"Anos de matricula: {df['ano_matricula'].min():.0f} - {df['ano_matricula'].max():.0f}")print(f"Registros con ano de matricula: {df['ano_matricula'].notna().sum():,}")print(f"Matriculas 2024-2025: {((df['ano_matricula'] >= 2024) & (df['ano_matricula'] <= 2025)).sum():,}")

## 3. SUBDIMENSION 1: Entrada Empresarial**Indicadores:**- Tasa de nuevas matriculas por comuna- Densidad de nuevas empresas por hectarea

In [ ]:
# --- TASA DE NUEVAS MATRICULAS POR COMUNA ---# Formula: (matriculas_2024_2025 / total_empresas_comuna) * 100# Interpretacion: Que porcentaje de las empresas actuales son "nuevas" (ultimos 2 anos).# Un valor alto indica dinamismo empresarial (mucha creacion reciente).# Un valor bajo indica un tejido empresarial maduro o estancado.nuevas = df_cc[df_cc['ano_matricula'] >= 2024].groupby(col_comuna).size().reset_index(name='nuevas_2024_2025')total = df_cc.groupby(col_comuna).size().reset_index(name='total_empresas')entrada = total.merge(nuevas, on=col_comuna, how='left')entrada['nuevas_2024_2025'] = entrada['nuevas_2024_2025'].fillna(0)entrada['tasa_nuevas_pct'] = (entrada['nuevas_2024_2025'] / entrada['total_empresas'] * 100).round(1)# --- DENSIDAD DE NUEVAS EMPRESAS POR HECTAREA ---# Formula: nuevas_matriculas_2024_2025 / area_comuna_hectareas# Interpretacion: Cuantas empresas nuevas se crearon por hectarea.# Normaliza por tamano del territorio para comparar comunas grandes y pequenas.if gdf_comunas is not None:    entrada = entrada.merge(gdf_comunas[['comuna_key', 'area_ha']], left_on=col_comuna, right_on='comuna_key', how='left')    entrada['densidad_nuevas_ha'] = (entrada['nuevas_2024_2025'] / entrada['area_ha']).round(2)entrada_sorted = entrada.sort_values('tasa_nuevas_pct', ascending=False)print("ENTRADA EMPRESARIAL - Tasa de nuevas matriculas por comuna (2024-2025)")print("=" * 90)print(f"{'Comuna':<12} {'Total':>8} {'Nuevas':>7} {'Tasa%':>6} {'Nuevas/ha':>10}")print("-" * 90)for _, r in entrada_sorted.head(22).iterrows():    dens = f"{r['densidad_nuevas_ha']:.2f}" if pd.notna(r.get('densidad_nuevas_ha')) else 'N/A'    print(f"{r[col_comuna]:<12} {int(r['total_empresas']):>8,} {int(r['nuevas_2024_2025']):>7,} {r['tasa_nuevas_pct']:>5.1f}% {dens:>10}")print("=" * 90)# Graficofig, axes = plt.subplots(1, 2, figsize=(16, 7))fig.suptitle('Subdimension: Entrada Empresarial (2024-2025)', fontsize=14, fontweight='bold')top = entrada_sorted.head(20)axes[0].barh(top[col_comuna], top['tasa_nuevas_pct'], color='#2ecc71', edgecolor='white')axes[0].set_title('Tasa de nuevas matriculas (%)\n(nuevas_2024_2025 / total) * 100')axes[0].set_xlabel('%')axes[0].invert_yaxis()if 'densidad_nuevas_ha' in entrada.columns:    top2 = entrada.sort_values('densidad_nuevas_ha', ascending=False).head(20)    axes[1].barh(top2[col_comuna], top2['densidad_nuevas_ha'].fillna(0), color='#3498db', edgecolor='white')    axes[1].set_title('Densidad nuevas empresas/ha\n(nuevas / area_hectareas)')    axes[1].set_xlabel('Empresas nuevas / ha')    axes[1].invert_yaxis()plt.tight_layout()fig.savefig(OUTPUT_DIR / 'ind_01_entrada_empresarial.png', dpi=150, bbox_inches='tight', facecolor='white')print('[OK] Guardado: ind_01_entrada_empresarial.png')plt.show()

## 4. SUBDIMENSION 2: Persistencia Empresarial**Indicadores:**- Tasa de renovacion 2025- Antiguedad promedio por comuna

In [ ]:
# --- TASA DE RENOVACION 2025 ---# Formula: (empresas_renovadas_2025 / total_empresas) * 100# Interpretacion: Que porcentaje de empresas renovaron su matricula en 2025.# Una tasa alta indica empresas activas y comprometidas con la formalidad.# Una tasa baja puede indicar fragilidad o informalidad creciente.renovadas_2025 = df_cc[df_cc['ano_renovacion'] == 2025].groupby(col_comuna).size().reset_index(name='renovadas_2025')persistencia = total.merge(renovadas_2025, on=col_comuna, how='left')persistencia['renovadas_2025'] = persistencia['renovadas_2025'].fillna(0)persistencia['tasa_renovacion_pct'] = (persistencia['renovadas_2025'] / persistencia['total_empresas'] * 100).round(1)# --- ANTIGUEDAD PROMEDIO POR COMUNA ---# Formula: 2025 - ano_matricula (promedio por comuna)# Interpretacion: Cuantos anos en promedio tienen las empresas de cada comuna.# Comunas con alta antiguedad tienen tejido empresarial maduro y estable.# Comunas con baja antiguedad son zonas de expansion reciente.antiguedad = df_cc[df_cc['ano_matricula'].notna()].copy()antiguedad['antiguedad'] = 2025 - antiguedad['ano_matricula']ant_comuna = antiguedad.groupby(col_comuna)['antiguedad'].mean().reset_index(name='antiguedad_promedio')ant_comuna['antiguedad_promedio'] = ant_comuna['antiguedad_promedio'].round(1)persistencia = persistencia.merge(ant_comuna, on=col_comuna, how='left')persistencia_sorted = persistencia.sort_values('tasa_renovacion_pct', ascending=False)print("PERSISTENCIA EMPRESARIAL")print("=" * 80)print(f"{'Comuna':<12} {'Total':>8} {'Renovadas':>10} {'Tasa Ren%':>10} {'Antiguedad':>11}")print("-" * 80)for _, r in persistencia_sorted.head(22).iterrows():    ant = f"{r['antiguedad_promedio']:.1f} anos" if pd.notna(r.get('antiguedad_promedio')) else 'N/A'    print(f"{r[col_comuna]:<12} {int(r['total_empresas']):>8,} {int(r['renovadas_2025']):>10,} {r['tasa_renovacion_pct']:>9.1f}% {ant:>11}")print("=" * 80)# Graficofig, axes = plt.subplots(1, 2, figsize=(16, 7))fig.suptitle('Subdimension: Persistencia Empresarial', fontsize=14, fontweight='bold')top = persistencia_sorted.head(20)axes[0].barh(top[col_comuna], top['tasa_renovacion_pct'], color='#1abc9c', edgecolor='white')axes[0].set_title('Tasa de renovacion 2025 (%)\n(renovadas_2025 / total) * 100')axes[0].set_xlabel('%')axes[0].invert_yaxis()top2 = persistencia.sort_values('antiguedad_promedio', ascending=False).head(20)axes[1].barh(top2[col_comuna], top2['antiguedad_promedio'].fillna(0), color='#e67e22', edgecolor='white')axes[1].set_title('Antiguedad promedio (anos)\n2025 - ano_matricula')axes[1].set_xlabel('Anos')axes[1].invert_yaxis()plt.tight_layout()fig.savefig(OUTPUT_DIR / 'ind_02_persistencia_empresarial.png', dpi=150, bbox_inches='tight', facecolor='white')print('[OK] Guardado: ind_02_persistencia_empresarial.png')plt.show()

## 5. SUBDIMENSION 3: Transformacion Sectorial**Indicadores:**- Participacion % por CIIU y comuna- Indice de diversificacion Shannon- Indice HHI (concentracion)- Empleo por sector

In [ ]:
# --- INDICE DE DIVERSIFICACION SHANNON ---# Formula: H = -SUM(pi * ln(pi)) donde pi = proporcion de cada CIIU en la comuna# Interpretacion:#   - H alto (>3): Economia muy diversificada, muchas actividades diferentes.#     El barrio/comuna no depende de un solo sector. Mas resiliente a crisis.#   - H bajo (<2): Economia concentrada en pocos sectores.#     Vulnerable si ese sector entra en crisis.# Rango tipico: 0 (una sola actividad) a ln(N) donde N = numero de actividades.# --- INDICE HHI (HERFINDAHL-HIRSCHMAN) ---# Formula: HHI = SUM(pi^2) donde pi = proporcion de cada CIIU# Interpretacion:#   - HHI cercano a 0: Muy diversificado (muchas empresas en muchos sectores)#   - HHI cercano a 1: Muy concentrado (pocas empresas dominan)# Es el indicador estandar de concentracion de mercado usado por reguladores.def calcular_shannon(grupo):    '''Calcula indice Shannon de diversificacion economica'''    conteos = grupo.value_counts()    proporciones = conteos / conteos.sum()    # Evitar log(0)    proporciones = proporciones[proporciones > 0]    return -(proporciones * np.log(proporciones)).sum()def calcular_hhi(grupo):    '''Calcula indice HHI de concentracion'''    conteos = grupo.value_counts()    proporciones = conteos / conteos.sum()    return (proporciones ** 2).sum()# Calcular por comunashannon = df_cc.groupby(col_comuna)[col_ciiu].apply(calcular_shannon).reset_index(name='shannon')hhi = df_cc.groupby(col_comuna)[col_ciiu].apply(calcular_hhi).reset_index(name='hhi')n_ciiu = df_cc.groupby(col_comuna)[col_ciiu].nunique().reset_index(name='n_ciiu')transformacion = total.merge(shannon, on=col_comuna).merge(hhi, on=col_comuna).merge(n_ciiu, on=col_comuna)transformacion['shannon'] = transformacion['shannon'].round(3)transformacion['hhi'] = transformacion['hhi'].round(4)# Top sector por comunatop_sector = df_cc.groupby(col_comuna)[col_sector].agg(lambda x: x.value_counts().index[0]).reset_index(name='sector_dominante')transformacion = transformacion.merge(top_sector, on=col_comuna, how='left')transformacion_sorted = transformacion.sort_values('shannon', ascending=False)print("TRANSFORMACION SECTORIAL - Diversificacion economica por comuna")print("=" * 100)print(f"{'Comuna':<12} {'CIIU':>5} {'Shannon':>8} {'HHI':>7} {'Sector dominante':<50}")print("-" * 100)for _, r in transformacion_sorted.head(22).iterrows():    sector = str(r['sector_dominante'])[:48] if pd.notna(r.get('sector_dominante')) else 'N/A'    print(f"{r[col_comuna]:<12} {int(r['n_ciiu']):>5} {r['shannon']:>8.3f} {r['hhi']:>7.4f} {sector:<50}")print("=" * 100)print(f"\nInterpretacion Shannon: >3.5 = muy diversificado | 2.5-3.5 = moderado | <2.5 = concentrado")print(f"Interpretacion HHI: <0.05 = diversificado | 0.05-0.15 = moderado | >0.15 = concentrado")

In [ ]:
# Grafico de transformacion sectorialfig, axes = plt.subplots(1, 3, figsize=(18, 7))fig.suptitle('Subdimension: Transformacion Sectorial', fontsize=14, fontweight='bold')top = transformacion_sorted.head(22)# Shannonaxes[0].barh(top[col_comuna], top['shannon'], color='#9b59b6', edgecolor='white')axes[0].set_title('Indice Shannon (diversificacion)\nH = -SUM(pi * ln(pi))\nMayor = mas diversificado')axes[0].set_xlabel('Shannon')axes[0].invert_yaxis()# HHI (invertido: menor = mejor)top_hhi = transformacion.sort_values('hhi', ascending=True).head(22)axes[1].barh(top_hhi[col_comuna], top_hhi['hhi'], color='#e74c3c', edgecolor='white')axes[1].set_title('Indice HHI (concentracion)\nHHI = SUM(pi^2)\nMenor = mas diversificado')axes[1].set_xlabel('HHI')axes[1].invert_yaxis()# N CIIUtop_ciiu = transformacion.sort_values('n_ciiu', ascending=False).head(22)axes[2].barh(top_ciiu[col_comuna], top_ciiu['n_ciiu'], color='#3498db', edgecolor='white')axes[2].set_title('Actividades CIIU distintas\n(conteo de codigos unicos)')axes[2].set_xlabel('CIIU')axes[2].invert_yaxis()plt.tight_layout()fig.savefig(OUTPUT_DIR / 'ind_03_transformacion_sectorial.png', dpi=150, bbox_inches='tight', facecolor='white')print('[OK] Guardado: ind_03_transformacion_sectorial.png')plt.show()

## 6. SUBDIMENSION 4: Intensidad Economica**Indicadores:**- Ingresos promedio por empresa- Ingresos promedio de empresas activas (excluye $0)- % empresas con ingresos > $0

In [ ]:
# --- INGRESOS PROMEDIO TOTAL ---# Formula: mean(ingresos) por comuna (incluye empresas con $0)# Interpretacion: Nivel economico general. Sesgado por empresas inactivas.# --- INGRESOS PROMEDIO DE ACTIVAS ---# Formula: mean(ingresos) WHERE ingresos > 0# Interpretacion: Nivel economico REAL de las empresas que operan.# Este es el indicador mas representativo porque excluye las que no reportan.# --- % EMPRESAS CON INGRESOS > $0 ---# Formula: (empresas_con_ingresos / total_empresas) * 100# Interpretacion: Que proporcion del tejido empresarial tiene actividad real.# Comunas con bajo % tienen muchas empresas "de papel" o inactivas.intensidad = df_cc.groupby(col_comuna).agg(    total_empresas=(col_comuna, 'size'),    ingresos_promedio=(col_ingresos, 'mean'),    ingresos_mediana=(col_ingresos, 'median'),).reset_index()# Solo activasactivas = df_cc[df_cc[col_ingresos] > 0]ing_activas = activas.groupby(col_comuna)[col_ingresos].mean().reset_index(name='ingresos_prom_activas')n_activas = activas.groupby(col_comuna).size().reset_index(name='n_con_ingresos')intensidad = intensidad.merge(ing_activas, on=col_comuna, how='left')intensidad = intensidad.merge(n_activas, on=col_comuna, how='left')intensidad['pct_con_ingresos'] = (intensidad['n_con_ingresos'] / intensidad['total_empresas'] * 100).round(1)intensidad_sorted = intensidad.sort_values('ingresos_prom_activas', ascending=False)print("INTENSIDAD ECONOMICA - Ingresos operacionales por comuna")print("=" * 90)print(f"{'Comuna':<12} {'Total':>8} {'Ing.Prom':>12} {'Ing.Activas':>12} {'%ConIng':>8} {'Mediana':>12}")print("-" * 90)for _, r in intensidad_sorted.head(22).iterrows():    ip = f"${r['ingresos_promedio']/1e6:.1f}M" if pd.notna(r['ingresos_promedio']) and r['ingresos_promedio'] >= 1e6 else "$0"    ia = f"${r['ingresos_prom_activas']/1e6:.1f}M" if pd.notna(r.get('ingresos_prom_activas')) and r['ingresos_prom_activas'] >= 1e6 else "$0"    med = f"${r['ingresos_mediana']/1e6:.1f}M" if pd.notna(r['ingresos_mediana']) and r['ingresos_mediana'] >= 1e6 else "$0"    pct = f"{r['pct_con_ingresos']:.0f}%" if pd.notna(r.get('pct_con_ingresos')) else "N/A"    print(f"{r[col_comuna]:<12} {int(r['total_empresas']):>8,} {ip:>12} {ia:>12} {pct:>8} {med:>12}")print("=" * 90)# Graficofig, axes = plt.subplots(1, 2, figsize=(16, 7))fig.suptitle('Subdimension: Intensidad Economica', fontsize=14, fontweight='bold')top = intensidad_sorted.head(20)axes[0].barh(top[col_comuna], top['ingresos_prom_activas'].fillna(0) / 1e6, color='#27ae60', edgecolor='white')axes[0].set_title('Ingreso promedio empresas activas ($M)\nmean(ingresos) WHERE ingresos > 0')axes[0].set_xlabel('$M')axes[0].invert_yaxis()top2 = intensidad.sort_values('pct_con_ingresos', ascending=False).head(20)axes[1].barh(top2[col_comuna], top2['pct_con_ingresos'].fillna(0), color='#2980b9', edgecolor='white')axes[1].set_title('% Empresas con ingresos > $0\n(n_con_ingresos / total) * 100')axes[1].set_xlabel('%')axes[1].invert_yaxis()plt.tight_layout()fig.savefig(OUTPUT_DIR / 'ind_04_intensidad_economica.png', dpi=150, bbox_inches='tight', facecolor='white')print('[OK] Guardado: ind_04_intensidad_economica.png')plt.show()

## 7. SUBDIMENSION 5: Empleo Formal**Indicadores:**- Empleos formales por comuna- Ratio hombres/mujeres- Empleo por 1000 habitantes (si hay datos demograficos)

In [ ]:
# --- EMPLEO FORMAL POR COMUNA ---# Formula: SUM(personal_ocupado) por comuna# Interpretacion: Total de empleos formales declarados en el Registro Mercantil.# --- RATIO HOMBRES/MUJERES ---# Formula: SUM(numero_de_hombres) / SUM(numero_de_mujeres)# Interpretacion:#   - Ratio > 1: Mas hombres empleados que mujeres#   - Ratio = 1: Paridad de genero en empleo#   - Ratio < 1: Mas mujeres empleadas que hombres# Util para medir brechas de genero en el empleo formal por territorio.# --- EMPLEO PROMEDIO POR EMPRESA ---# Formula: SUM(personal_ocupado) / total_empresas# Interpretacion: Tamano promedio de las empresas en terminos de empleo.empleo = df_cc.groupby(col_comuna).agg(    total_empresas=(col_comuna, 'size'),    empleo_total=(col_empleo, 'sum'),    empleo_promedio=(col_empleo, 'mean'),    hombres_total=(col_hombres, 'sum'),    mujeres_total=(col_mujeres, 'sum'),).reset_index()# Ratio H/M (evitar division por 0)empleo['ratio_hm'] = np.where(    empleo['mujeres_total'] > 0,    (empleo['hombres_total'] / empleo['mujeres_total']).round(2),    np.nan)# % mujeres del totalempleo['pct_mujeres'] = (empleo['mujeres_total'] / (empleo['hombres_total'] + empleo['mujeres_total']) * 100).round(1)empleo_sorted = empleo.sort_values('empleo_total', ascending=False)print("EMPLEO FORMAL por comuna")print("=" * 95)print(f"{'Comuna':<12} {'Empresas':>8} {'Empleo':>8} {'Emp/Emp':>8} {'Hombres':>8} {'Mujeres':>8} {'Ratio H/M':>10} {'%Mujeres':>9}")print("-" * 95)for _, r in empleo_sorted.head(22).iterrows():    ratio = f"{r['ratio_hm']:.2f}" if pd.notna(r['ratio_hm']) else "N/A"    print(f"{r[col_comuna]:<12} {int(r['total_empresas']):>8,} {int(r['empleo_total']):>8,} {r['empleo_promedio']:>8.1f} {int(r['hombres_total']):>8,} {int(r['mujeres_total']):>8,} {ratio:>10} {r['pct_mujeres']:>8.1f}%")print("=" * 95)# Graficofig, axes = plt.subplots(1, 3, figsize=(18, 7))fig.suptitle('Subdimension: Empleo Formal', fontsize=14, fontweight='bold')top = empleo_sorted.head(20)axes[0].barh(top[col_comuna], top['empleo_total'], color='#8e44ad', edgecolor='white')axes[0].set_title('Empleo total por comuna\nSUM(personal_ocupado)')aplicar_formato(axes[0], 'x')axes[0].invert_yaxis()# Ratio H/Mtop2 = empleo.sort_values('ratio_hm', ascending=True).head(20)colors = ['#e74c3c' if r > 1.5 else '#f39c12' if r > 1 else '#2ecc71' for r in top2['ratio_hm'].fillna(1)]axes[1].barh(top2[col_comuna], top2['ratio_hm'].fillna(0), color=colors, edgecolor='white')axes[1].axvline(1.0, color='black', linestyle='--', alpha=0.5, label='Paridad (1.0)')axes[1].set_title('Ratio Hombres/Mujeres\nhombres / mujeres\n(1.0 = paridad)')axes[1].legend(fontsize=9)axes[1].invert_yaxis()# % Mujerestop3 = empleo.sort_values('pct_mujeres', ascending=False).head(20)axes[2].barh(top3[col_comuna], top3['pct_mujeres'], color='#e91e63', edgecolor='white')axes[2].axvline(50, color='black', linestyle='--', alpha=0.5, label='50%')axes[2].set_title('% Mujeres en empleo formal\nmujeres / (hombres+mujeres) * 100')axes[2].set_xlabel('%')axes[2].legend(fontsize=9)axes[2].invert_yaxis()plt.tight_layout()fig.savefig(OUTPUT_DIR / 'ind_05_empleo_formal.png', dpi=150, bbox_inches='tight', facecolor='white')print('[OK] Guardado: ind_05_empleo_formal.png')plt.show()

## 8. Tabla resumen consolidada de todos los indicadores

In [ ]:
# Consolidar todos los indicadores en una tablaconsolidado = total.copy()consolidado = consolidado.merge(entrada[['comuna', 'tasa_nuevas_pct', 'densidad_nuevas_ha']].rename(columns={'comuna': col_comuna}) if 'densidad_nuevas_ha' in entrada.columns else entrada[[col_comuna, 'tasa_nuevas_pct']], on=col_comuna, how='left')consolidado = consolidado.merge(persistencia[[col_comuna, 'tasa_renovacion_pct', 'antiguedad_promedio']], on=col_comuna, how='left')consolidado = consolidado.merge(transformacion[[col_comuna, 'shannon', 'hhi', 'n_ciiu']], on=col_comuna, how='left')consolidado = consolidado.merge(intensidad[[col_comuna, 'ingresos_prom_activas', 'pct_con_ingresos']], on=col_comuna, how='left')consolidado = consolidado.merge(empleo[[col_comuna, 'empleo_total', 'ratio_hm', 'pct_mujeres']], on=col_comuna, how='left')consolidado_sorted = consolidado.sort_values('total_empresas', ascending=False)print("TABLA CONSOLIDADA - TODOS LOS INDICADORES PROPUESTOS")print("=" * 130)print(f"{'Comuna':<12} {'Emp':>6} {'%Nuevas':>7} {'%Renov':>7} {'Antig':>6} {'Shannon':>8} {'HHI':>6} {'IngAct$M':>9} {'%ConIng':>7} {'Empleo':>7} {'%Muj':>5}")print("-" * 130)for _, r in consolidado_sorted.head(22).iterrows():    ia = f"{r['ingresos_prom_activas']/1e6:.0f}" if pd.notna(r.get('ingresos_prom_activas')) and r['ingresos_prom_activas'] >= 1e6 else "0"    print(f"{r[col_comuna]:<12} {int(r['total_empresas']):>6,} {r['tasa_nuevas_pct']:>6.1f}% {r['tasa_renovacion_pct']:>6.1f}% {r['antiguedad_promedio']:>5.1f} {r['shannon']:>8.3f} {r['hhi']:>6.4f} {ia:>9} {r['pct_con_ingresos']:>6.1f}% {int(r['empleo_total']):>7,} {r['pct_mujeres']:>4.1f}%")print("=" * 130)

## 9. Generar reporte .txt

In [ ]:
REPORTE = OUTPUT_DIR / 'indicadores_propuestos_actividad_economica.txt'with open(REPORTE, 'w', encoding='utf-8') as f:    f.write('=' * 100 + '\n')    f.write('INDICADORES PROPUESTOS - DIMENSION ACTIVIDAD ECONOMICA\n')    f.write('Registro Mercantil 2025 - Santiago de Cali\n')    f.write(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')    f.write('=' * 100 + '\n\n')        f.write('SUBDIMENSIONES IMPLEMENTADAS:\n')    f.write('  1. Entrada empresarial: tasa_nuevas_pct, densidad_nuevas_ha\n')    f.write('  2. Persistencia empresarial: tasa_renovacion_pct, antiguedad_promedio\n')    f.write('  3. Transformacion sectorial: shannon, hhi, n_ciiu\n')    f.write('  4. Intensidad economica: ingresos_prom_activas, pct_con_ingresos\n')    f.write('  5. Empleo formal: empleo_total, ratio_hm, pct_mujeres\n\n')        f.write('-' * 100 + '\n')    f.write('FORMULAS UTILIZADAS\n')    f.write('-' * 100 + '\n')    f.write('  tasa_nuevas_pct = (matriculas_2024_2025 / total_empresas) * 100\n')    f.write('  densidad_nuevas_ha = nuevas_matriculas / area_hectareas\n')    f.write('  tasa_renovacion_pct = (renovadas_2025 / total_empresas) * 100\n')    f.write('  antiguedad_promedio = mean(2025 - ano_matricula)\n')    f.write('  shannon = -SUM(pi * ln(pi)) [diversificacion]\n')    f.write('  hhi = SUM(pi^2) [concentracion]\n')    f.write('  ingresos_prom_activas = mean(ingresos) WHERE ingresos > 0\n')    f.write('  pct_con_ingresos = (empresas_con_ingresos / total) * 100\n')    f.write('  ratio_hm = hombres / mujeres\n')    f.write('  pct_mujeres = mujeres / (hombres + mujeres) * 100\n\n')        f.write('-' * 100 + '\n')    f.write('TABLA CONSOLIDADA\n')    f.write('-' * 100 + '\n')    f.write(consolidado_sorted.drop(columns=['densidad_nuevas_ha'], errors='ignore').to_string(index=False))    f.write('\n\n')        f.write('=' * 100 + '\n')    f.write('FIN\n')    f.write('=' * 100 + '\n')print(f'[OK] Reporte: {REPORTE.name}')

## 10. Notas**Subdimensiones implementadas (5 de 7):**1. Entrada empresarial - Tasa de nuevas matriculas y densidad por hectarea2. Persistencia empresarial - Tasa de renovacion y antiguedad promedio3. Transformacion sectorial - Shannon, HHI, conteo CIIU4. Intensidad economica - Ingresos activas, % con ingresos5. Empleo formal - Total, ratio H/M, % mujeres**No implementables sin datos adicionales:**- Mortalidad/Salida (requiere datos de cancelaciones)- Escalamiento empresarial (requiere datos longitudinales)**Formulas clave:**- Shannon: `H = -SUM(pi * ln(pi))` — mayor = mas diversificado- HHI: `SUM(pi^2)` — menor = mas diversificado- Tasa nuevas: `(nuevas_2024_2025 / total) * 100`- Ratio H/M: `hombres / mujeres` — 1.0 = paridad